# Kaggle Submission — DINOv3 + SigLIP2 Ensemble (CSE 144 Final Project)

Inference-only notebook. Loads pre-trained checkpoints and writes `submission.csv`.

- **DINOv3 ViT-L/16 + LoRA** member: backbone built offline from uploaded raw weights, LoRA
  + MLP head restored from the trained checkpoint.
- **SigLIP2** member: optional second ensemble member (see the SigLIP note below).
- Per-member **test-time augmentation** (5 transforms, softmax-averaged), then a weighted
  average of the members' probabilities (`ProbEnsemble`, equal weights).

This notebook is **self-contained** — it does not import the project `src/` package, so it
runs as-is on Kaggle.


## 1. Install dependencies

In [ ]:
!pip install -q "timm>=1.0.20" "peft>=0.7.0" "transformers>=4.45.0" "omegaconf>=2.3.0"

## 2. Paths & settings

`SIGLIP_CHECKPOINT_PATH` is a **placeholder** — SigLIP2 is not uploaded yet. The notebook
checks whether the file exists: if it is missing, the run proceeds with DINOv3 only; once you
upload the SigLIP2 checkpoint and set the path, it automatically becomes a 2-model ensemble.


In [ ]:
COMPETITION = "ucsc-cse-144-spring-2026-final-project"
TEST_DIR = f"/kaggle/input/competitions/{COMPETITION}/test"

# DINOv3
WEIGHTS_PATH = "/kaggle/input/models/allexsmith/dinov3-raw/pytorch/default/1/dinov3_vitl16.pth"
CHECKPOINT_PATH = "/kaggle/input/models/allexsmith/new-best-dino-lora/pytorch/default/1/best_new.pth"

# SigLIP2
SIGLIP_BACKBONE_SOURCE = "/kaggle/input/models/allexsmith/"
SIGLIP_CHECKPOINT_PATH = "/kaggle/input/models/allexsmith/"

OUTPUT_PATH = "/kaggle/working/submission.csv"

USE_TTA = True

## 3. Imports

In [ ]:
import os
import random
from abc import ABC, abstractmethod
from typing import List, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.nn.init import trunc_normal_
from torchvision import transforms
from tqdm.auto import tqdm

## 4. Determinism & device

In [ ]:
def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
        return "mps"
    return "cpu"


set_seed(42)
DEVICE = get_device()
print(f"Device: {DEVICE}")

## 5. Backbones, head, LoRA

Mirrors the project `src/` definitions exactly so checkpoint `state_dict` keys line up.
DINOv3 is built with `pretrained=False` (offline-safe architecture only) and its weights are
loaded from the uploaded raw-weights file. SigLIP2 is wrapped so `get_image_features` returns
the pooled `[B, D]` vision embedding.


In [ ]:
import timm
from transformers import AutoModel


class BackboneWrapper(ABC, nn.Module):
    @abstractmethod
    def forward_features(self, x: torch.Tensor) -> torch.Tensor: ...

    @property
    @abstractmethod
    def embed_dim(self) -> int: ...


class TimmBackbone(BackboneWrapper):
    def __init__(self, model: nn.Module, dim: int):
        super().__init__()
        self.model = model
        self._embed_dim = dim

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        features = self.model.forward_features(x)
        if features.dim() == 3:  # timm ViT returns [B, N, D]; take CLS token
            features = features[:, 0]
        return features

    @property
    def embed_dim(self) -> int:
        return self._embed_dim


class Siglip2Backbone(BackboneWrapper):
    def __init__(self, source: str):
        super().__init__()
        # `source` is an HF id (needs internet) OR an uploaded local HF model dir (offline).
        self.model = AutoModel.from_pretrained(source)
        self._embed_dim = self.model.config.vision_config.hidden_size

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        out = self.model.get_image_features(pixel_values=x)
        # transformers version drift: some return the pooled [B, D] tensor directly, others
        # return a BaseModelOutputWithPooling — unwrap to the pooled image embedding.
        if isinstance(out, torch.Tensor):
            return out
        return out.pooler_output

    @property
    def embed_dim(self) -> int:
        return self._embed_dim


class LinearHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int = 100):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
        trunc_normal_(self.fc.weight, std=0.02)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        return self.fc(x)


class MLPHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 512, num_classes: int = 100, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                trunc_normal_(m.weight, std=0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


def build_head(head_type: str, in_dim: int, num_classes: int = 100, **kwargs) -> nn.Module:
    if head_type == "linear":
        return LinearHead(in_dim, num_classes)
    if head_type == "mlp":
        return MLPHead(in_dim, num_classes=num_classes, **kwargs)
    raise ValueError(f"Unknown head type: {head_type}")


def _find_target_modules(model: nn.Module, target_blocks: int) -> List[str]:
    all_names = [name for name, _ in model.named_modules()]
    patterns = [
        ("attn.qkv",),
        ("attn.q_proj", "attn.v_proj"),
        ("attention.query", "attention.value"),
    ]
    block_ids = []
    for name in all_names:
        for prefix in ("blocks.", "encoder.layer."):
            if prefix in name:
                idx = name.split(prefix)[1].split(".")[0]
                if idx.isdigit():
                    block_ids.append(int(idx))
    if not block_ids:
        return []
    max_block = max(block_ids)
    targets = set(range(max_block - target_blocks + 1, max_block + 1))
    for pat_group in patterns:
        matches = []
        for name in all_names:
            for pat in pat_group:
                if pat in name:
                    for prefix in ("blocks.", "encoder.layer."):
                        if prefix in name:
                            idx = int(name.split(prefix)[1].split(".")[0])
                            if idx in targets:
                                matches.append(name)
        if matches:
            return matches
    return []


def apply_lora(backbone: nn.Module, r: int, alpha: int, target_blocks: int) -> nn.Module:
    from peft import LoraConfig, get_peft_model

    inner = backbone.model if hasattr(backbone, "model") else backbone
    target_modules = _find_target_modules(inner, target_blocks)
    if not target_modules:
        raise ValueError("Could not find LoRA target modules in backbone.")
    config = LoraConfig(
        r=r, lora_alpha=alpha, target_modules=target_modules, lora_dropout=0.05, bias="none"
    )
    for p in inner.parameters():
        p.requires_grad = True
    backbone.model = get_peft_model(inner, config)
    return backbone


class FewShotClassifier(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone.forward_features(x)
        return self.head(features)


class FewShotClassifierWithLoRA(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        features = self.backbone.forward_features(x)
        return self.head(features)

## 6. Transforms, TTA, ensemble, test dataset

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# SigLIP normalization is fixed (mean=std=0.5); size comes from the checkpoint config.
SIGLIP_MEAN = (0.5, 0.5, 0.5)
SIGLIP_STD = (0.5, 0.5, 0.5)


def build_tta_transforms(
    image_size: int,
    mean: Sequence[float] = IMAGENET_MEAN,
    std: Sequence[float] = IMAGENET_STD,
) -> List[transforms.Compose]:
    normalize = transforms.Normalize(mean=mean, std=std)
    base = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        normalize,
    ])
    hflip = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        normalize,
    ])
    scale_crops = []
    for size in (image_size - 16, image_size - 8, image_size + 16):
        scale_crops.append(transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            normalize,
        ]))
    return [base, hflip] + scale_crops


@torch.no_grad()
def predict_with_tta(model, image, tta_transforms, device) -> torch.Tensor:
    model.eval()
    probs_sum = None
    for tfm in tta_transforms:
        img = tfm(image).unsqueeze(0).to(device)
        probs = F.softmax(model(img), dim=1).squeeze(0)
        probs_sum = probs if probs_sum is None else probs_sum + probs
    return (probs_sum / len(tta_transforms)).cpu()


class ProbEnsemble:
    def __init__(self, weights: Sequence[float]):
        if not weights:
            raise ValueError("ProbEnsemble needs at least one weight")
        self.weights = [float(w) for w in weights]

    @torch.no_grad()
    def combine(self, member_probs: Sequence[torch.Tensor]) -> torch.Tensor:
        if len(member_probs) != len(self.weights):
            raise ValueError(f"Expected {len(self.weights)} members, got {len(member_probs)}")
        total = sum(self.weights)
        combined = sum(w * p for w, p in zip(self.weights, member_probs))
        return combined / total


class TestImageDataset:
    def __init__(self, root: str):
        self.root = root
        self.image_ids = sorted(
            int(os.path.splitext(f)[0])
            for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        print(f"Loaded {len(self.image_ids)} test images from {root}")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        path = os.path.join(self.root, f"{image_id}.jpg")
        return Image.open(path).convert("RGB"), image_id

## 7. Member loaders


In [ ]:
def _load_state(model: nn.Module, state: dict) -> None:
    if state and all(k.startswith("head.") for k in state):
        head_state = {k[len("head."):]: v for k, v in state.items()}
        model.head.load_state_dict(head_state)
    else:
        model.load_state_dict(state)


def load_dino_member(checkpoint_path: str, weights_path: str, device: str) -> dict:
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]

    timm_model = timm.create_model(
        "vit_large_patch16_dinov3.lvd1689m", pretrained=False, num_classes=0
    )
    # Offline backbone weights (raw timm state_dict saved during training setup).
    timm_model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    backbone = TimmBackbone(timm_model, timm_model.num_features)

    head_kwargs = {}
    if cfg["head"]["type"] == "mlp":
        head_kwargs = {
            "hidden": cfg["head"].get("hidden", 512),
            "dropout": cfg["head"].get("dropout", 0.2),
        }
    head = build_head(cfg["head"]["type"], backbone.embed_dim, cfg["num_classes"], **head_kwargs)

    if cfg.get("lora", {}).get("enabled", False):
        backbone = apply_lora(
            backbone, cfg["lora"]["r"], cfg["lora"]["alpha"], cfg["lora"]["target_blocks"]
        )
        model = FewShotClassifierWithLoRA(backbone, head)
    else:
        model = FewShotClassifier(backbone, head)

    _load_state(model, ckpt["state_dict"])
    model.eval().to(device)

    image_size = cfg.get("image_size", 256)
    print(f"DINOv3 member: epoch={ckpt.get('epoch')}, val_acc={ckpt.get('val_acc')}, "
          f"image_size={image_size}, lora={cfg.get('lora', {}).get('enabled', False)}")
    return {
        "model": model,
        "transforms": build_tta_transforms(image_size, IMAGENET_MEAN, IMAGENET_STD),
    }


def load_siglip_member(checkpoint_path: str, device: str) -> dict:
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg = ckpt["config"]

    source = cfg["backbone"].get("model_name", "google/siglip2-so400m-patch14-384")
    source = globals().get("SIGLIP_BACKBONE_SOURCE", source)
    backbone = Siglip2Backbone(source)

    head_kwargs = {}
    if cfg["head"]["type"] == "mlp":
        head_kwargs = {
            "hidden": cfg["head"].get("hidden", 512),
            "dropout": cfg["head"].get("dropout", 0.2),
        }
    head = build_head(cfg["head"]["type"], backbone.embed_dim, cfg["num_classes"], **head_kwargs)
    model = FewShotClassifier(backbone, head)

    _load_state(model, ckpt["state_dict"])
    model.eval().to(device)

    # SigLIP's position embeddings are fixed to its native patch grid, so the input MUST be
    # the model's native resolution (e.g. 384 for so400m-patch14-384), NOT the project
    # image_size (256). Read it from the vision config so it's always correct.
    image_size = backbone.model.config.vision_config.image_size
    print(f"SigLIP2 member: epoch={ckpt.get('epoch')}, val_acc={ckpt.get('val_acc')}, "
          f"image_size={image_size}, source={source}")
    return {
        "model": model,
        "transforms": build_tta_transforms(image_size, SIGLIP_MEAN, SIGLIP_STD),
    }

## 8. Load ensemble members

In [ ]:
members = []

members.append(load_dino_member(CHECKPOINT_PATH, WEIGHTS_PATH, DEVICE))
members.append(load_siglip_member(SIGLIP_CHECKPOINT_PATH, DEVICE))

weights = [1.0] * len(members)
ensemble = ProbEnsemble(weights)
print(f"\nEnsembling {len(members)} model(s) with weights {weights} (TTA={USE_TTA})")

## 9. Predict & write submission

In [ ]:
test_dataset = TestImageDataset(TEST_DIR)

ids, labels = [], []
for idx in tqdm(range(len(test_dataset)), desc="Predicting"):
    image, image_id = test_dataset[idx]
    if USE_TTA:
        member_probs = [predict_with_tta(m["model"], image, m["transforms"], DEVICE)
                        for m in members]
    else:
        member_probs = []
        for m in members:
            tfm = m["transforms"][0]  # direct-resize transform only
            img = tfm(image).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                member_probs.append(F.softmax(m["model"](img), dim=1).squeeze(0).cpu())
    combined = ensemble.combine(member_probs)
    ids.append(image_id)
    labels.append(int(combined.argmax().item()))

df = pd.DataFrame({"ID": ids, "Label": labels}).sort_values("ID").reset_index(drop=True)
df["ID"] = df["ID"].astype(str) + ".jpg"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Submission saved to {OUTPUT_PATH} ({len(df)} rows)")

## 10. Inspect submission

In [ ]:
df = pd.read_csv(OUTPUT_PATH)
assert list(df.columns) == ["ID", "Label"], f"Bad columns: {list(df.columns)}"
assert df["ID"].str.endswith(".jpg").all(), "IDs must be formatted as '<n>.jpg'"
ids_num = df["ID"].str.replace(".jpg", "", regex=False).astype(int)
assert ids_num.is_monotonic_increasing, "IDs must be sorted ascending"
assert ids_num.tolist() == list(range(len(df))), "IDs must be contiguous 0..N-1"
assert df["Label"].between(0, 99).all(), "Labels must be in 0..99"
print(f"Rows: {len(df)} | ID range: {ids_num.min()}-{ids_num.max()} | "
      f"unique labels: {df['Label'].nunique()}")
print(df.head(10).to_string(index=False))